# Clustering of Cases

**Research question.** Do the cases fall into a few groups that run through the process in clearly different ways?

**Hypothesis.** K-means finds a small number of clearly separated groups of cases, and these groups differ in how long they take.

## 1. Create Dataset

Idea: Cluster cases by their amount of offers, validation rounds, and waiting time for the customer. The clusters should explain the outcome and throughput time

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from preprocessing import load_raw_log

BLUE, AQUA, ORANGE, GRAY, MUTED = '#2a78d6', '#1baf7a', '#eb6834', '#c3c2b7', '#52514e'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': GRAY, 'axes.titlelocation': 'left',
                     'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.labelcolor': MUTED, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

raw = load_raw_log()

# Filter out workflow events and only keep the start of each activity
events = raw[(raw['EventOrigin'] != 'Workflow') | (raw['lifecycle:transition'] == 'start')]
activity = events['concept:name']

# create a dataframe with one row per case, including the start/end, amount of offers, validation rounds and time waiting for the customer
cases = events.groupby('case:concept:name').agg(start=('time:timestamp', 'min'), end=('time:timestamp', 'max'))
cases['offers'] = events[activity == 'O_Create Offer'].groupby('case:concept:name').size().reindex(cases.index, fill_value=0)
cases['validations'] = events[activity == 'A_Validating'].groupby('case:concept:name').size().reindex(cases.index, fill_value=0)

first_sent = events[activity.str.startswith('O_Sent')].groupby('case:concept:name')['time:timestamp'].min().reindex(cases.index)
first_returned = events[activity == 'O_Returned'].groupby('case:concept:name')['time:timestamp'].min().reindex(cases.index)
waited_until = first_returned.fillna(cases['end'])


cases['customer_days'] = ((waited_until - first_sent).dt.total_seconds() / 86400).fillna(0)

FEATURES = ['offers', 'validations', 'customer_days']
cases[FEATURES].describe().round(1)

In [ ]:
cases

## 2. Preprocessing


In [ ]:
# log-transform to reduce the effect of outliers and scale to zero mean and unit variance
X = StandardScaler().fit_transform(np.log1p(cases[FEATURES]))

## 3. Number of clusters

In [ ]:
# K-means needs initial cluster number -> Silhouette score, computed on a random sample of 5000 cases

sample = np.random.default_rng(42).choice(len(X), 5000, replace=False)
silhouette = pd.Series({k: silhouette_score(X[sample], KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).labels_[sample])
                        for k in range(2, 7)})

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(silhouette.index, silhouette.to_numpy(), color=BLUE, linewidth=2, marker='o')
ax.axhline(0.5, color=GRAY, linewidth=1, linestyle='--')
ax.set_xlabel('number of clusters k')
ax.set_title('Silhouette score per k')
fig.savefig('../output/figures/clustering_silhouette.png')
plt.show()
silhouette.round(3)

Note: Although k=4 has a slightly higher score than k=3, k=3 was chosen, because the fourth cluster (8% of the cases) only splits the No reply cases by their number of offers (one offer vs. two or more) and adds unnecessary complexity

## 4. Result

In [ ]:

# Apply K-means with k=3
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)

# Transform the cluster centers back to the original scale
scaler = StandardScaler().fit(np.log1p(cases[FEATURES]))
centers = pd.DataFrame(np.expm1(scaler.inverse_transform(kmeans.cluster_centers_)), columns=FEATURES)


def name(center):
    if center['validations'] < 0.5:
        return 'No reply'  # the customer did not return the offer, so there was nothing to validate
    return 'Negotiation' if center['offers'] >= 1.5 else 'Standard'

# Output the cluster centers and the mean offers, validations, customer waiting days and the share of cases in each cluster
centers.index = centers.apply(name, axis=1)
cases['cluster'] = centers.index[kmeans.labels_]
ORDER = ['Standard', 'Negotiation', 'No reply']
centers['share of cases'] = cases['cluster'].value_counts(normalize=True)
centers.loc[ORDER].round(2)

In [ ]:
COLORS = {'Standard': BLUE, 'Negotiation': AQUA, 'No reply': ORANGE}
center = centers.to_dict('index')  # plain numbers per cluster
shown = cases.iloc[sample]
rng = np.random.default_rng(0)
# Many cases share the same values, so the dots are spread slightly to keep them visible
x_spread, y_spread = rng.uniform(-1, 1, len(shown)), rng.uniform(-0.3, 0.3, len(shown))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
for ax, feature in zip(axes, ['validations', 'offers']):
    for cluster in ORDER:
        own = (shown['cluster'] == cluster).to_numpy()
        ax.scatter(shown['customer_days'][own] + x_spread[own], shown[feature][own] + y_spread[own],
                   s=6, alpha=0.35, color=COLORS[cluster], linewidths=0)
    for cluster in ORDER:
        ax.scatter(center[cluster]['customer_days'], center[cluster][feature], s=180, color=COLORS[cluster],
                   edgecolor='white', linewidth=2, zorder=3)
    ax.set_xlim(0, 60)
    ax.set_xlabel('days waiting for the customer')
    ax.set_ylabel(f'number of {feature}')
axes[0].set_title('Validation rounds separate No reply from the others')
axes[1].set_title('Offers separate Negotiation from Standard')
handles = [Line2D([], [], marker='o', linestyle='', markersize=8, color=COLORS[cluster],
                  label=f"{cluster} ({center[cluster]['share of cases']:.0%})") for cluster in ORDER]
fig.legend(handles=handles, ncol=3, frameon=False, loc='lower left', bbox_to_anchor=(0.05, 0.97))
fig.savefig('../output/figures/clustering_clusters.png')
plt.show()

Problem with 2D: It looks like there are points in wrong clusters, but they are actually in the right cluster when looking at all three features. The 3D figure shows that the clusters are well separated

In [ ]:
from typing import cast

from mpl_toolkits.mplot3d import Axes3D

z_spread = rng.uniform(-0.3, 0.3, len(shown))
fig = plt.figure(figsize=(8, 6.5))
ax3d = cast(Axes3D, fig.add_subplot(projection='3d'))
for cluster in ORDER:
    own = (shown['cluster'] == cluster).to_numpy()
    # plot without lines draws the dots like a scatter plot
    ax3d.plot(shown['customer_days'][own] + x_spread[own], shown['validations'][own] + y_spread[own],
              shown['offers'][own] + z_spread[own], linestyle='', marker='o', markersize=2.5, markeredgewidth=0,
              alpha=0.45, color=COLORS[cluster], axlim_clip=True)
for cluster in ORDER:
    ax3d.plot([center[cluster]['customer_days']], [center[cluster]['validations']], [center[cluster]['offers']], linestyle='',
              marker='o', markersize=15, markeredgecolor='white', markeredgewidth=2, color=COLORS[cluster])
# The limits leave room for the spread of the dots, so that only cases outside the range are cut off
ax3d.set_xlim(-1, 60)
ax3d.set_ylim(-0.4, 6.4)
ax3d.set_zlim(0.6, 5.4)
ax3d.zaxis.set_ticks(range(1, 6))
ax3d.set_xlabel('days waiting for the customer')
ax3d.set_ylabel('number of validations')
ax3d.set_zlabel('number of offers')
ax3d.view_init(elev=25, azim=-120)
ax3d.set_title('K-Means clustering of cases ')
fig.legend(handles=handles, ncol=3, frameon=False, loc='lower left', bbox_to_anchor=(0.08, 0.9))
fig.savefig('../output/figures/clustering_clusters_3d.png')
plt.show()

## 5. Results

In [ ]:
cases['throughput_days'] = (cases['end'] - cases['start']).dt.total_seconds() / 86400
outcome = events[activity.str.startswith('A_')].groupby('case:concept:name')['concept:name'].last()
cases['outcome'] = outcome.reindex(cases.index)

meaning = cases.groupby('cluster')['throughput_days'].median().rename('median throughput days').to_frame()
meaning = meaning.join(pd.crosstab(cases['cluster'], cases['outcome'], normalize='index')[['A_Pending', 'A_Cancelled', 'A_Denied']])
meaning.loc[ORDER].round(2)